# KoChatGPT 파이프라인(SFT+RM+PPO)을 카카오브레인 KoGPT 백본에 적용한 RLHF 학습

!!! 주의  
계속 작업중임 (3/13 금 오후2시 기준)

- Backbone: kakaobrain/kogpt (기본 언어 능력)  

- Weights: KoChatGPT (SFT/RM/PPO 학습 결과물)  

카카오 모델을 가져와서 KoChatGPT라는 다운스트림 태스크(챗봇화)를 수행했다.  

#### 추가사항
1. 학습 에포크 조절
2. 양자화 시도
3. LoRA

## 1. 준비

In [ ]:
# cuda 설치 버전 확인 - 런타임이 GPU여야 실행됨
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [ ]:
# 코랩-CUDA 12.8빌드 기준 (cuda 설치 버전에 따라 아래 버전들이 달라짐.)

# 지금 환경(torch 2.10.0 + CUDA 12.8)에 맞는 정확한 조합
!pip install -q bitsandbytes==0.49.2  # CUDA 12.8 공식 지원
!pip install -q transformers==4.45.0 --force-reinstall
!pip install -q accelerate sentencepiece python-dotenv huggingface_hub

# 시간 엄청 많이 걸림. but, 모델 로딩 시 에러 나면 이것도 실행할 것.
# !pip install -q torch==2.10.0+cu128 --index-url https://download.pytorch.org/whl/cu128

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.0/802.0 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.2/507.2 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78

라이브러리들 설치 후 커널 재시작 필요.  
재시작 이후에는 다시 설치할 필요없음.

In [ ]:
# 라이브러리 버전 확인
import torch
import bitsandbytes as bnb
import transformers

"""
== BEST ==
torch: 2.10.0+cu128
bitsandbytes: 0.49.2
transformers: 4.45.0
CUDA available: True
"""

print(f"torch: {torch.__version__}")
print(f"bitsandbytes: {bnb.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

torch: 2.10.0+cu128
bitsandbytes: 0.49.2
transformers: 4.45.0
CUDA available: True


In [ ]:
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device) # True가 나와야 GPU 사용 가능

# 1. 병렬 처리 경고 방지
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 2. PyTorch 최적화 (2.2+ 기능을 활용해 추론 속도를 높임)
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

cuda


In [ ]:
# 구글 드라이브 마운트 & 소스파일 가져오기
from google.colab import drive
drive.mount('/content/drive')
project_name = 'kogpt_upgrade'
mypath = os.path.join('/content/drive/MyDrive/Aiffel/project', project_name)
os.makedirs(mypath, exist_ok=True)  # 프로젝트 폴더 없으면 생성

GIT_DIR = os.path.join(mypath, "KoChatGPT")
MODEL_DIR = os.path.join(mypath, "model")
DATA_DIR = os.path.join(mypath, "data")

# KoChatGPT 깃 클론
!git clone https://github.com/airobotlab/KoChatGPT {mypath}

# 마운트한 구글 드라이브에 git clone한 폴더 경로
gpt_dir = os.path.join(mypath, "KoChatGPT/colossalai_ChatGPT_230319")

Mounted at /content/drive
fatal: destination path '/content/drive/MyDrive/Aiffel/project/kogpt_upgrade' already exists and is not an empty directory.


In [ ]:
# 허깅페이스 인증
from dotenv import load_dotenv
from huggingface_hub import login

# 1. .env 파일 로드
load_dotenv(os.path.join(mypath, ".env"))

# 2. 환경 변수에서 토큰 가져오기
HF_TOKEN = os.getenv("HF_TOKEN")

# 3. 허깅페이스 로그인 (권한 인증)
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Hugging Face 인증 성공!")
else:
    print("에러: .env 파일에서 HF_TOKEN을 찾을 수 없습니다.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face 인증 성공!


In [ ]:
# 긴 텍스트 출력시 화면 밖으로 안넘치게 보여주는 함수 정의
def print2(r):
    s = 0
    step = 80
    for i in range(0, len(r), step):
        if len(r) - i > step:
            print(r[s:s + step])
            s += step
        else:
            print(r[s:])

In [ ]:
# 메모리 청소 함수
def vram_status():
    print(f"VRAM 총량: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"VRAM 사용 중: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"VRAM 여유: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1024**3:.2f} GB")
    return

def clean_memory():
    import gc

    # VRAM 확인
    print("----- GPU 메모리 청소전 -----")
    vram_status()

    # 기존에 모델이 있다면 메모리에서 제거 (del 명령어도 가능)
    tokenizer = None
    model = None
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    !rm -rf /root/.cache/huggingface/hub/models--kakaobrain--kogpt # HuggingFace 캐시 파일 삭제(모델 관련 문제 제거)

    print("\n---------- 청소후 ----------")
    vram_status()

    return

clean_memory()

----- GPU 메모리 청소전 -----
VRAM 총량: 14.6 GB
VRAM 사용 중: 0.00 GB
VRAM 여유: 14.56 GB

---------- 청소후 ----------
VRAM 총량: 14.6 GB
VRAM 사용 중: 0.00 GB
VRAM 여유: 14.56 GB


#### 메모리 청소 함수 설명

1. `del model` VS `model = None`  
    결과는 동일. 둘다 해당 객체의 참조를 끊는 것.   
    이후 gc.collect()가 메모리를 해제한다.  
    - 메모리 해제 -> "이 메모리 공간을 다시 써도 된다"고 OS에 반환하는 것.

2. HuggingFace 캐시 삭제  
    다음 실행 시 모델을 처음부터 다시 다운로드하게 강제한다.  
    메모리 확보와는 직접적인 관계는 없다.  
    아래 경우에 사용한다.  
    - 다운로드 중 끊기거나 디스크 오류로 파일이 깨진 경우, 캐시를 그대로 쓰면 모델 로딩 자체가 실패하거나 이상하게 동작한다.  
        이 때 깨끗하게 지우고 다시 받으려고 사용.
    - 모델이 업데이트 되었는데 캐시에 구버전 모델이 저장되어 있을 때.  
        캐시 삭제 후 재다운로드해서 최신 버전을 받을 수 있다.
    - 모델 하나가 수십 GB일 경우 안쓰는 캐시를 지워서 디스크 용량을 확보할 수 있다.
    
3. `torch.cuda.synchronize()`  
    GPU는 기본적으로 비동기로 작동한다. 즉, CPU가 GPU에 명령을 전달하고 기다리지 않고 다음 코드를 실행하는 방식.  
    이 메서드를 실행하면 현재 작업이 진행되는 동안 GPU가 현재 que에 있는 모든 작업을 완료할 때까지 CPU를 블로킹하게 된다.  
    즉, CPU가 GPU의 작업이 끝나기를 기다리게 된다.

    여기서는 단지 GPU작업이 완전히 끝난 상태에서 VRAM 수치를 읽게 하려고 쓰였다.  
    지금 시점의 GPU작업이 끝나면 자동으로 다시 비동기로 동작하기 때문에 원복하거나 할 필요가 없다.  
    GPU의 설정을 바꾸는 게 아님. 일시정지에 가깝다.

4. `gc.collect()` VS `empty_cache()`
    - `gc.collect()`: CPU(RAM) 쪽 Python 객체를 정리.   
    - `empty_cache()`: GPU VRAM은 PyTorch가 별도 관리한다.   
        PyTorch는 성능을 위해 해제된 VRAM을 OS에 반환하지 않고 캐시로 들고 있는데, 이 캐시를 비워 지금 당장 VRAM 여유공간을 확보하는 것.


## 2. Model Dataset for RLHF

In [ ]:
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM

print(f"import 후 VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

# 1. 토크나이저 먼저 로드
tokenizer = AutoTokenizer.from_pretrained('kakaobrain/kogpt', revision='KoGPT6B-ryan1.5b-float16')
# 2. 스페셜 토큰 추가 (필요한 경우)
special_tokens_dict = {
    'bos_token': '[BOS]',
    'eos_token': '[EOS]',
    'unk_token': '[UNK]',
    'pad_token': '[PAD]',
    'mask_token': '[MASK]'
}

tokenizer.add_special_tokens(special_tokens_dict)

print(f"토크나이저 로드 후 VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

import 후 VRAM: 0.00 GB


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

토크나이저 로드 후 VRAM: 0.00 GB


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
# 기존 사전 크기 대비 늘어난 크기 확인
print(f"{tokenizer.name_or_path} 사전의 총 길이: {len(tokenizer)}")

# 추가한 스페셜 토큰이 어떤 ID를 부여받았는지 확인
print(f"BOS 토큰 ID: {tokenizer.bos_token_id}")
print(f"PAD 토큰 ID: {tokenizer.pad_token_id}")

# 실제 텍스트 인코딩 테스트
test_text = "[BOS] 인공지능 학습 시작 [EOS]"
print(tokenizer.encode(test_text))

kakaobrain/kogpt 사전의 총 길이: 63998
BOS 토큰 ID: 0
PAD 토큰 ID: 3
[0, 7692, 2542, 1157, 327, 1]


#### from_pretrained 내부 동작
1. pytorch_model.bin 다운로드 ← 성공 (12.3G 다운로드 완료)
2. 가중치 로드
3. 양자화 적용
4. 모델을 device에 배치 ← 여기서 .to() 호출하다 실패. GPU 메모리 터짐.

In [ ]:
# 모델 불러오기 (양자화 적용)
from transformers import BitsAndBytesConfig
import torch

clean_memory()
print()

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True
)
model = AutoModelForCausalLM.from_pretrained(
    'kakaobrain/kogpt',
    revision='KoGPT6B-ryan1.5b-float16',  # or float32 version: revision=KoGPT6B-ryan1.5b
    pad_token_id=tokenizer.pad_token_id,
    low_cpu_mem_usage=True,
    quantization_config=bnb_config,      # 양자화 -> 가중치를 8비트 정수로 압축해서 로드 (VRAM 약 7~8GB 예상), 이 파라미터는 객체로 넘겨져야 해서 bnb_config사용
    device_map="auto",       # 자동으로 GPU/CPU 분산 로드
)

print()
vram_status()

# _ = model.eval()

----- GPU 메모리 청소전 -----
VRAM 총량: 14.6 GB
VRAM 사용 중: 0.00 GB
VRAM 여유: 14.56 GB

---------- 청소후 ----------
VRAM 총량: 14.6 GB
VRAM 사용 중: 0.00 GB
VRAM 여유: 14.56 GB



config.json:   0%|          | 0.00/839 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/12.3G [00:00<?, ?B/s]

Some weights of the model checkpoint at kakaobrain/kogpt were not used when initializing GPTJForCausalLM: ['transformer.h.0.attn.bias', 'transformer.h.0.attn.masked_bias', 'transformer.h.1.attn.bias', 'transformer.h.1.attn.masked_bias', 'transformer.h.10.attn.bias', 'transformer.h.10.attn.masked_bias', 'transformer.h.11.attn.bias', 'transformer.h.11.attn.masked_bias', 'transformer.h.12.attn.bias', 'transformer.h.12.attn.masked_bias', 'transformer.h.13.attn.bias', 'transformer.h.13.attn.masked_bias', 'transformer.h.14.attn.bias', 'transformer.h.14.attn.masked_bias', 'transformer.h.15.attn.bias', 'transformer.h.15.attn.masked_bias', 'transformer.h.16.attn.bias', 'transformer.h.16.attn.masked_bias', 'transformer.h.17.attn.bias', 'transformer.h.17.attn.masked_bias', 'transformer.h.18.attn.bias', 'transformer.h.18.attn.masked_bias', 'transformer.h.19.attn.bias', 'transformer.h.19.attn.masked_bias', 'transformer.h.2.attn.bias', 'transformer.h.2.attn.masked_bias', 'transformer.h.20.attn.bias'


VRAM 총량: 14.6 GB
VRAM 사용 중: 6.24 GB
VRAM 여유: 8.32 GB


In [ ]:
prompt = '인간처럼 생각하고, 행동하는 \'지능\'을 통해 인류가 이제까지 풀지 못했던'
with torch.no_grad():
    tokens = tokenizer.encode(prompt, return_tensors='pt').to('cuda:0')  # device_map="auto"이면 이렇게
    gen_tokens = model.generate(tokens, do_sample=True, temperature=0.8, max_length=64)
    generated = tokenizer.batch_decode(gen_tokens)[0]
print(f"{model.name_or_path} 모델 로딩 완료!")

print2(generated) # print: 인간처럼 생각하고, 행동하는 '지능'을 통해 인류가 이제까지 풀지 못했던 문제의 해답을 찾을 수 있을 것이다. 과학기술이 고도로 발달한 21세기를 살아갈 우리 아이들에게 가장 필요한 것은 사고력 훈련이다. 사고력 훈련을 통해, 세상

kakaobrain/kogpt 모델 로딩 완료!
인간처럼 생각하고, 행동하는 '지능'을 통해 인류가 이제까지 풀지 못했던 난제(難題)를 해결할 수 있을지 주목된다. 뇌 연구를 통해 인류의 미래
를 바꿀 차세대 기술이 탄생할지도 모를 일. 과학계에서는 인간을 더욱 똑똑하게


현재까지 문제점... GPU 메모리(VRAM) 용량 이슈로 kakaobrain model을 load할 수가 없음.
결국 그래도 해냄 ㅠㅠ  
1. 모델 자체가 fp32로 저장되어 있어서 float16 옵션이 무시됨(?) -> 양자화 불가...한 줄 알았으나 되네?
2. 최적의 라이브러리들 버전 조합을 찾느라 4시간 걸림

## 양자화 전 후 비교

아래 코드는 양자화 적용하지 않고 모델 로딩함. 비교용.

In [ ]:
# # 양자화 적용 안함
# # VRAM 확인
# print(f"VRAM 사용 중: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
# print(f"예약 VRAM: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

# # 모델 불러오기 (양자화 없음)
# model = AutoModelForCausalLM.from_pretrained(
#     'kakaobrain/kogpt',
#     revision='KoGPT6B-ryan1.5b-float16',
#     pad_token_id=tokenizer.pad_token_id,
#     torch_dtype=torch.float16,  # 양자화 대신 float16으로 로드
#     low_cpu_mem_usage=True,
#     device_map="auto",
# )
# _ = model.eval()

In [ ]:
def check_model_memory(model):
    """모델의 양자화 상태 및 메모리 사용량을 출력하는 함수"""

    # 1. dtype별 레이어 개수
    dtype_count = {}
    for name, param in model.named_parameters():
        dtype = str(param.dtype)
        dtype_count[dtype] = dtype_count.get(dtype, 0) + 1

    print("=" * 50)
    print("[레이어 개수]")
    total = sum(dtype_count.values())
    for dtype, count in dtype_count.items():
        print(f"  {dtype}: {count}개 ({count/total*100:.1f}%)")

    # 2. dtype별 실제 용량
    print("\n[파라미터 용량]")
    dtype_size = {}
    for name, param in model.named_parameters():
        dtype = str(param.dtype)
        size_mb = param.numel() * param.element_size() / 1024**2
        dtype_size[dtype] = dtype_size.get(dtype, 0) + size_mb

    total_param_mb = sum(dtype_size.values())
    for dtype, size in dtype_size.items():
        print(f"  {dtype}: {size:.1f} MB ({size/total_param_mb*100:.1f}%)")
    print(f"  합계: {total_param_mb/1024:.2f} GB")

    # 3. VRAM 사용량
    print("\n[VRAM 사용량]")
    vram_allocated = torch.cuda.memory_allocated() / 1024**3
    vram_reserved  = torch.cuda.memory_reserved() / 1024**3
    overhead       = vram_allocated - total_param_mb / 1024
    print(f"  파라미터 용량:    {total_param_mb/1024:.2f} GB")
    print(f"  실제 VRAM 사용:   {vram_allocated:.2f} GB")
    print(f"  런타임 오버헤드:  {overhead:.2f} GB")
    print(f"  예약 VRAM:        {vram_reserved:.2f} GB")

    # 4. 양자화 여부 요약
    print("\n[양자화 요약]")
    if 'torch.int8' in dtype_size:
        int8_ratio = dtype_size['torch.int8'] / total_param_mb * 100
        print(f"  양자화 적용됨 (int8 비율: {int8_ratio:.1f}%)")
    elif 'torch.uint8' in dtype_size:
        uint8_ratio = dtype_size['torch.uint8'] / total_param_mb * 100
        print(f"  양자화 적용됨 (4bit/uint8 비율: {uint8_ratio:.1f}%)")
    else:
        print("  양자화 없음 (전체 float16/float32)")
    print("=" * 50)

In [ ]:
check_model_memory(model)

[레이어 개수]
  torch.float16: 117개 (41.1%)
  torch.int8: 168개 (58.9%)

[파라미터 용량]
  torch.float16: 1009.7 MB (15.8%)
  torch.int8: 5376.0 MB (84.2%)
  합계: 6.24 GB

[VRAM 사용량]
  파라미터 용량:    6.24 GB
  실제 VRAM 사용:   6.26 GB
  런타임 오버헤드:  0.03 GB
  예약 VRAM:        6.35 GB

[양자화 요약]
  양자화 적용됨 (int8 비율: 84.2%)


### 양자화 전 후 결과

**양자화 전**  
```text
==================================================
[레이어 개수]
  torch.float16: 285개 (100.0%)

[파라미터 용량]
  torch.float16: 11761.7 MB (100.0%)
  합계: 11.49 GB

[VRAM 사용량]
  파라미터 용량:    11.49 GB
  실제 VRAM 사용:   11.60 GB
  런타임 오버헤드:  0.11 GB
  예약 VRAM:        11.60 GB

[양자화 요약]
  양자화 없음 (전체 float16/float32)
==================================================
```

**양자화 후**  
```text
==================================================
[레이어 개수]
  torch.float16: 117개 (41.1%)
  torch.int8: 168개 (58.9%)

[파라미터 용량]
  torch.float16: 1009.7 MB (15.8%)
  torch.int8: 5376.0 MB (84.2%)
  합계: 6.24 GB

[VRAM 사용량]
  파라미터 용량:    6.24 GB
  실제 VRAM 사용:   6.26 GB
  런타임 오버헤드:  0.03 GB
  예약 VRAM:        6.35 GB

[양자화 요약]
  양자화 적용됨 (int8 비율: 84.2%)
==================================================
```
---
**왜 레이어 개수 비율과 용량 비율이 다른가?**  
레이어 개수 비율(41% vs 58%)과 용량 비율(15% vs 84%)이 다른 이유
-> 레이어마다 파라미터 크기가 천차만별이기 때문.  
LayerNorm, bias 같은 레이어는 개수는 많지만 크기가 매우 작고,  
어텐션/MLP 같은 레이어는 개수는 적지만 크기가 매우 큽니다.    

**무엇을 봐야 하나?**  
양자화의 목적이 "메모리를 줄이는 것"이므로 실제 용량이 더 중요.  
용량 기준으로는 84.2%가 int8로 양자화된 것.  
개수 비율(58.9%)보다 용량 비율(84.2%)이 훨씬 높은 것은  
크기가 큰 레이어들이 집중적으로 압축됐다는 뜻.

#### 결론
```
transformer.wte.weight: torch.float16  
transformer.h.0.ln_1.weight: torch.float16  
transformer.h.0.ln_1.bias: torch.float16  
transformer.h.0.attn.k_proj.weight: torch.int8  
transformer.h.0.attn.v_proj.weight: torch.int8  
```
---

8비트 양자화가 모든 레이어에 적용되진 않았음.  
`wte.weight (임베딩): float16 유지 → 양자화 안 함`  
`ln_1.weight, ln_1.bias (LayerNorm): float16 유지 → 양자화 안 함`  
`attn.k_proj.weight, attn.v_proj.weight (어텐션): torch.int8 → 양자화 적용`  

LayerNorm, 임베딩, bias 같은 레이어는 의도적으로 float16으로 유지하는 게 bitsandbytes 8비트 양자화의 설계입니다.  
이 레이어들을 int8로 바꾸면 오히려 성능이 크게 떨어짐.

In [ ]:
import pandas as pd

input_txt = "바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까."
tokens = tokenizer(input_txt).tokens()
input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].numpy()

pd.options.display.max_columns = 40
pd.options.display.max_rows = 60
df = pd.DataFrame([tokens, input_ids[0]], index=["kakaobrain/kogpt", "Input_IDs"])
df

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
kakaobrain/kogpt,ë°ĶëŀĮ,ëıĦ,ĠìĹĨ,ëĬĶ,Ġê³µì¤ĳ,ìĹĲ,ĠìĪĺì§ģ,ìĿĺ,ĠíĮĮë¬¸,ìĿĦ,ĠëĤ´ìĿ´,ë©°,Ġê³łìļĶ,íŀĪ,Ġëĸ¨ìĸ´ì§Ģ,ëĬĶ,Ġìĺ¤ëıĻ,ìŀİ,ìĿĢ,ĠëĪĦêµ¬,ìĿĺ,Ġë°ľìŀĲì·¨,ĠìŀħëĭĪê¹Į,.
Input_IDs,7214,418,640,378,5272,379,10009,383,17192,387,62359,566,10881,654,4245,378,32669,5642,409,2411,383,36542,44552,120


## RLHF전 생성 성능 점검

In [ ]:
input_txt = "바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까."
input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].to(device)
max_length = 128

#### 일반 생성

In [ ]:
# 테스트 생성 및 디코딩 성능 확인
output_greedy = model.generate(input_ids, max_length=max_length, do_sample=False)
print2(tokenizer.decode(output_greedy[0]))

바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까. 지리한 장마 끝에 서풍에 몰려가는 무서운 검은 구름의 
터진 틈으로 언뜻언뜻 보이는 푸른 하늘은 누구의 얼굴입니까. 꽃도 없는 깊은 나무에 푸른 이끼를 거쳐서 옛 탑 위의 고요한 하늘을 스치는 알 수
 없는 향기는 누구의 입김입니까. 근원은 알지도 못할 곳에서 나서 돌부리를 울리고 가늘게 흐르는 작은 시내는 굽이굽이 누구의 노래입니까. 연꽃 
같은 발꿈치로 가이 없는 바다를 밟고


#### 일반 생성 - base 모델 결과

```text
바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까.'
"그렇다면 그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리
```

#### 빔 서치 디코딩, n-gram 패널티 부과

In [ ]:
output_beam = model.generate(input_ids, max_length=max_length, num_beams=10, no_repeat_ngram_size=2,
                             do_sample=False)
print2(tokenizer.decode(output_beam[0]))

바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까. 지리한 장마 끝에 서풍에 몰려가는 무서운 검은 구름의 
터진 틈으로 언뜻언뜻 보이며 푸른 하늘에 닿을 듯이 세월에 불타고 우뚝 남아 서서 차라리 봄도 꽃피진 말아라. 낡은 거미집 휘두르고 끝없는 꿈길
에 혼자 설레이는 마음은 아예 뉘우침 아니라. 검은 그림자 쓸쓸하면 마침내 호수 속 깊이 거꾸러져 차마 바람도 흔들진 못해라. 요점 정리 지은이
: 이은상(李殷相) 갈래: 자유시. 서정시 율


#### 빔 서치 디코딩, n-gram 패널티 부과 - base 모델 결과

```text
바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까.'
"그렇지 않습니다."
"어떻게 된 일입니까?"
그녀는 고개를 갸웃거렸다.
"아니, 그게 무슨 말씀이신지 모르겠습니다만."
"무슨 말씀인지 알 수가 없군요."
아무런 대답도 하지 않은 채 그녀는 고개를 끄덕였다.
"그래, 알았어."
그녀의 눈에서 눈물이 주르륵 흘러내렸다.
그녀가 다시 입을 열었다.
"정말 죄송합니다, 고마워요, 고맙습니다"
"
```

### 빔서치 + 샘플링, top_k

In [ ]:
# top_k 샘플링: 전체 vocab중 확률 상위 k개 토큰만 후보로 남기고 나머지는 버림
# 아래 코드는 그 k개 안에서 temperature로 조정된 확률로 샘플링함.
output_beam = model.generate(input_ids, max_length=max_length, num_beams=7, no_repeat_ngram_size=2,
                             do_sample=True, temperature=2.0, top_k=50)
print2(tokenizer.decode(output_beam[0]))

바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까. 지리한 장마 끝에 서풍에 몰려가는 무서운 검은 구름의 
터진 틈으로 언뜻언뜻 보이며 가는 푸른 하늘에 무거운 긴장을 내어 푸는 그는 누구인가 밤배를 저어서가는 수부들의 등불은 등활(等闊)한 공간 속에
 일렁이는 물결 위에 불멸의 유수(流水)를 던집니다. 사랑하는 나의 산하여! 나의 조국이여! 나는 오늘도 기다리고 있습니다. 둥기둥 줄이 울면 
초가 삼


#### 빔서치 + 샘플링, top_k - base 모델 결과

```text
바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까.'
"그럼, 그게 어디서 온 건가요?"
"네, 저는 그쪽에서 왔기 때문에."
"아니, 내가 그쪽으로 온 것은 아니라서."
한참이 지나서야 나는 입가에 미소가 번졌다.
"그래, 한 번 봤으면 좋겠어."
그 말에 나는 고개를 끄덕였다.
아무 생각 없이 나를 흘겨보면서.
"그러면 내일 아침 9시 30분에 아침을 먹어요."
그 말을 듣고 난 나는 괜히 기분이 상한 것
```

### 빔서치 + 샘플링, top_p

In [ ]:
# top_p 샘플링: 확률이 높은 토큰부터 누적해서 합이 top_p 값(여기서 0.90 = 90%)이 될 때까지의 토큰들만 후보로 남김
output_beam = model.generate(input_ids, max_length=max_length, num_beams=7, no_repeat_ngram_size=2,
                             do_sample=True, top_p=0.90)
print2(tokenizer.decode(output_beam[0]))

바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까. 지리한 장마 끝에 서풍에 몰려가는 무서운 검은 구름의 
터진 틈으로 언뜻언뜻 보이며 푸른 하늘에 닿을 듯이 세월에 불타고 우뚝 남아 서서 차라리 봄도 꽃피진 말아라. 낡은 거미집 휘두르고 끝없는 꿈길
에 혼자 설레이는 마음은 아예 뉘우침 아니라. 검은 그림자 쓸쓸하면 마침내 호수 속 깊이 거꾸러져 차마 바람도 흔들진 못해라. 요점 정리 지은이
: 이은상(李殷相) 갈래: 자유시. 서정시 율


#### 빔서치 + 샘플링, top_p - base 모델 결과

```text
바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까. 그건 너의 것이 아니란다."
"그렇다면 너는 누구인가?"
그녀는 고개를 갸우뚱거렸다.
"아니야. 나는 너를 사랑한 적이 없단다. 네가 나를 사랑할 줄은 꿈에도 생각지 못했다."
나는 고개를 끄덕였다.
"그래. 하지만 난 너에게 사랑하지 않았어. 난 널 사랑했어."
"사랑하지 않으면 안 되는 거야."
그녀의 목소리에 나는 깜짝 놀랐다.
"난 너에 대해 아무것도 알지 못해.
```

## 데이터셋 확인

SFT시도할 이니셜 모델에 쓸 데이터셋

In [ ]:
import json
# /content/drive/MyDrive/Aiffel/project/kogpt_upgrade/KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl
data_path_1_SFT = os.path.join(mypath, 'KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl')

with open(data_path_1_SFT, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

print(len(list_data_dict))
list_data_dict[:3]

12000


[{'prompt': '불고기용 고기 한우에요?',
  'completion': "'저는 인공지능 챗봇이며, 직접적으로 식품에 관한 정보를 가지고 있지 않습니다. 하지만 일반적으로 불고기용 고기는 한우, 쇠고기, 돼지고기 등 다양한 종류의 고기를 사용합니다. 하지만 한우는 대표적인 고급 육류로 알려져 있기 때문에, 한우를 사용하는 경우도 많습니다. 알러지나 개별 건강 상태에 따라 다를 수 있으니 충분한 정보 수집 후에 선택해 주시기 바랍니다.",
  'tokens': 193},
 {'prompt': '쓰던 앱이 유료로 전환됐어',
  'completion': "'어떤 앱인지 모르기 때문에 정확한 답변을 드리기 어렵습니다. 하지만, 일반적으로 유료 전환된 앱은 기존 무료 앱에서 추가적인 기능이나 서비스를 제공하는 경우가 많습니다. 이 경우, 유료 전환 이전 가격이 매우 저렴하거나 무료인 경우가 많으므로, 앱을 계속 사용하려면 추가적인 비용을 지불해야 할 수 있습니다. 하지만, 유료 전환 이후에는 기존 사용자 또는 새로운 사용자 모두가 추가 기능과 높은 품질의 서비스를 이용할 수 있게 됩니다. 따라서, 앱 개발자는 유료 모델로 전환함으로써 앱의 수익을 증가시키고 더 나은 서비스를 제공할 수 있습니다.",
  'tokens': 288},
 {'prompt': '여친이랑 다툼',
  'completion': "'저는 인공지능이기 때문에 여친과의 다툼에 대해서는 조언을 드릴 수는 없습니다. 하지만 다투게 되면 상대방의 말을 경청하는 태도와 서로의 감정을 존중하는 것이 중요합니다. 논리적이고 차분한 대화를 통해 문제를 해결해보세요. 그리고 서로를 이해하고 서로의 의견을 수용하는 것이 중요합니다.",
  'tokens': 153}]

RM에 사용할 데이터셋

In [ ]:
data_path_2_RM = os.path.join(mypath, 'KoChatGPT/data_kochatgpt/kochatgpt_2_RM.jsonl')
with open(data_path_2_RM, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

print(len(list_data_dict))
list_data_dict[:3]

10220


[{'prompt': '번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?',
  'completion_0': 'Allow me to answer your question. I know that you are curious about me.',
  'completion_1': '번디는 다양한 인터뷰자들과 뉴스홍보 담당자들과의 면담 때 밝혔다.',
  'completion_2': '라이언에게 말했다.',
  'ranking': [2, 1, 0]},
 {'prompt': '개포주공아파트는 몇 단지로 이루어져 있나?',
  'completion_0': '개포주공아파트는 다섯 단지로 이루어져 있습니다.',
  'completion_1': '이날 목송에서 구글상위노',
  'completion_2': '개포주공아파트는 총 27개 단지로 이루어져 있습니다.',
  'ranking': [2, 0, 1]},
 {'prompt': '김영삼의 후보 시절 지역표심을 겨냥한 발언을 문제삼은 후보는?',
  'completion_0': 'The diameter of the Metallic domain is bigger than the Hyperonic domain.',
  'completion_1': '이 질문은 조금 불분명합니다. 김영삼 대통령이 후보 시절에 어떤 발언을 했고, 누가 그 발언을 문제삼았는지에 따라 답이 다를 수 있습니다.\\n\\n만약 김영삼 대통령이 후보 시절에 지역표심을 겨냥한 발언을 했다는 가정하에, 그 발언을 문제삼은 후보가 누구였는지를 대답하자면, 그 답은 이화선 당시 민주당 대통령 후보가 될 것입니다. 1992년 총선 때, 김영삼 대선후보는 "집값이 오른 노량진역 부근의 부동산 가격은 세월호 폭침 후 \\\'강남 도시재생\\\' 일환으로 상승했다"는 발언을 했습니다. 하지만 이화선 후보는 이 발언을 "전국적으로 경제적 발전이 이루어지지 않은 지방민의 마음을 멀리해지려는 무례한 발언"이라고 비판하며 문

PPO 학습에 쓰일 데이터

In [ ]:
data_path_3_PPO = os.path.join(mypath, 'KoChatGPT/data_kochatgpt/kochatgpt_3_PPO.jsonl')
with open(data_path_3_PPO, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

print(len(list_data_dict))
list_data_dict[:3]

12000


[{'prompt': '번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?'},
 {'prompt': '개포주공아파트는 몇 단지로 이루어져 있나?'},
 {'prompt': '김영삼의 후보 시절 지역표심을 겨냥한 발언을 문제삼은 후보는?'}]

## 3. Supervised Fine-Tuning

기존 kogpt-2에서 kakaobrain/gpt로 변경

In [ ]:
from typing import Optional, Dict, Sequence
from torch.utils.data import Dataset
from dataclasses import dataclass
import logging
import copy

"""
kakaobrain의 토크나이저는 max_length가 기본적으로 엄청나게 큰 수이다.
연산과정에서 int변환 불가로 OverflowError를 발생시키니 아래와 같이 줄여준다.
kakaobrain 모델의 실제 최대 컨텍스트 길이는 2048이지만
skt/kogpt2는 512이므로 512로 지정해도 무방하다.
길어질수록 메모리 연산량이 증가한다.
"""
tokenizer.model_max_length = 512
print(tokenizer)
print("-" * 50)
print(tokenizer.decode(tokenizer("안녕하세요")["input_ids"]))
"""
<tokenizer>

Returns:
    dict: 다음 키를 포함
        - input_ids (Tensor): 토큰 ID 시퀀스. shape=(batch_size, seq_len)
        - attention_mask (Tensor): 패딩 토큰 구분 마스크. shape=(batch_size, seq_len)
                                   실제 토큰=1, 패딩 토큰=0
"""
print(tokenizer("안녕하세요", return_tensors="pt"))

PreTrainedTokenizerFast(name_or_path='kakaobrain/kogpt', vocab_size=63733, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '[BOS]', 'eos_token': '[EOS]', 'unk_token': '[UNK]', 'pad_token': '[PAD]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("[BOS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[EOS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	5: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	6: AddedToken("[SEP]", rstr

In [ ]:
# 모델 훈련을 위한 SFT 데이터셋 클래스
# 질문과 정답이 모두 포함된 데이터 -> 모델이 읽기 좋은 숫자(input_ids)로 변환
class SFT_dataset(Dataset):

    def __init__(self, data_path_1_SFT: str, tokenizer: transformers.PreTrainedTokenizer, verbose=False):
        super(SFT_dataset, self).__init__()
        logging.warning("Loading data...")

        pattern_instruction = 'prompt'  # instruction
        pattern_output = 'completion'  # response

        with open(data_path_1_SFT, "r", encoding='utf-8-sig') as json_file:
            list_data_dict = json.load(json_file)

        # 모델 인퍼런스 단계에서 사용할 prompt 딕셔너리 템플릿
        # 모델이 학습했던 것과 똑같은 말투/형식으로 질문을 던지기 위함
        PROMPT_DICT = {
            "prompt_input": (
                "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
            )
        }
        prompt_input = PROMPT_DICT["prompt_input"]

        sources = []
        for example in list_data_dict:
            tmp = prompt_input.format_map(example)
            sources.append(tmp)

        targets = []
        for example in list_data_dict:
            targets.append(f"{example[pattern_output]}{tokenizer.eos_token}")
        examples = [s + t for s, t in zip(sources, targets)]

        sources_tokenized = self._tokenize_fn(sources, tokenizer)  # source
        examples_tokenized = self._tokenize_fn(examples, tokenizer)  # source + target

        input_ids = examples_tokenized["input_ids"]
        labels = copy.deepcopy(input_ids)
        for label, source_len in zip(labels, sources_tokenized["input_ids_lens"]):
            label[:source_len] = -100

        data_dict = dict(input_ids=input_ids, labels=labels)

        self.input_ids = data_dict["input_ids"]
        self.labels = data_dict["labels"]
        logging.warning("Loading data done!!: %d"%(len(self.labels)))


    def _tokenize_fn(self, strings: Sequence[str], tokenizer: transformers.PreTrainedTokenizer) -> Dict:
        tokenized_list = [
            tokenizer(
                text,
                return_tensors="pt",
                padding="longest",
                max_length=tokenizer.model_max_length,
                truncation=True,
            )
            for text in strings
        ]
        input_ids = labels = [tokenized.input_ids[0] for tokenized in tokenized_list]
        input_ids_lens = labels_lens = [
            tokenized.input_ids.ne(tokenizer.pad_token_id).sum().item() for tokenized in tokenized_list
        ]
        return dict(
            input_ids=input_ids,
            labels=labels,
            input_ids_lens=input_ids_lens,
            labels_lens=labels_lens,
        )


    def __len__(self):
        return len(self.input_ids)


    def __getitem__(self, i) -> Dict[str, torch.Tensor]:
        return dict(input_ids=self.input_ids[i], labels=self.labels[i])

In [ ]:
@dataclass
class DataCollatorForSupervisedDataset(object):

    tokenizer: transformers.PreTrainedTokenizer

    def __call__(self, instances: Sequence[Dict]) -> Dict[str, torch.Tensor]:
        input_ids, labels = tuple([instance[key] for instance in instances] for key in ("input_ids", "labels"))
        input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids, batch_first=True, padding_value=self.tokenizer.pad_token_id
        )
        labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value= -100)
        return dict(
            input_ids=input_ids,
            labels=labels,
            attention_mask=input_ids.ne(self.tokenizer.pad_token_id),
        )

SFT_dataset 클래스를 사용해 훈련셋 만들기  
data collator 인스턴스 생성

In [ ]:
import numpy as np

# skt-KoChatGPT의 SFT_dataset 클래스 사용해서 훈련셋 생성
train_dataset_raw = SFT_dataset(
    data_path_1_SFT,
    tokenizer=tokenizer
)

class SFT_Subset:
    def __init__(self, dataset, indices):
        self.dataset = dataset
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        # 섞인 인덱스를 바탕으로 원본에서 데이터를 가져옴
        return self.dataset[self.indices[idx]]

    def __getattr__(self, name):
        # 나(Subset)한테 없는 속성(예: input_ids)을 찾으면
        # 원본 데이터셋(self.dataset)에게 가서 찾아오라는 뜻입니다.
        return getattr(self.dataset, name)

# --- 데이터 분리 로직 ---
full_dataset = SFT_dataset(data_path_1_SFT, tokenizer=tokenizer)

total_len = len(full_dataset)
indices = np.arange(total_len)
np.random.seed(42)
np.random.shuffle(indices)

train_size = int(0.9 * total_len)
train_indices = indices[:train_size]
eval_indices = indices[train_size:]

# skt 데이터셋임.
# train_dataset = SFT_Subset(full_dataset, train_indices)  # 전체 약 8시간 소요됨.
# eval_dataset = SFT_Subset(full_dataset, eval_indices)
train_dataset = SFT_Subset(full_dataset, train_indices[:1000])  #
eval_dataset = SFT_Subset(full_dataset, eval_indices[:250])

print(f"✅ 속성 연결 완료! 학습용: {len(train_dataset)} / 평가용: {len(eval_dataset)}")
print(f"✅ 수정 완료: 학습용 {len(train_dataset)} / 평가용 {len(eval_dataset)}")

"""
data collator
    -> 길이가 제각각 다른 데이터셋 샘플들을 가장 긴 샘플길이를 기준으로 패딩(PAD토큰)을 채워 길이를 맞춰줌
    입력: 데이터로더의 배치 단위 샘플들
    출력: 패딩으로 길이 맞춘 샘플들의 배치 행렬
"""
data_collator = DataCollatorForSupervisedDataset(tokenizer=tokenizer)

✅ 속성 연결 완료! 학습용: 1000 / 평가용: 250
✅ 수정 완료: 학습용 1000 / 평가용 250


In [ ]:
print(list_data_dict[0])  # 원본 데이터 텍스트
print(tokenizer.special_tokens_map)  # bos/eos 토큰 확인
print("\n--- 토큰 조각 확인 ---")
tokens = tokenizer.convert_ids_to_tokens(train_dataset.input_ids[0].tolist())

# 한글이 깨져보이지만 kakaobrain/kogpt의 BPE처리 때문으로, 실제로 깨진 것은 아니다.
# UTF-8 바이트 단위로 변환되었기 때문.
for idx, (id_, tok) in enumerate(zip(train_dataset.input_ids[0].tolist(), tokens)):
    print(f"[{idx:>3}] ID: {id_:>6}  |  {tok}")

{'prompt': '번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?'}
{'bos_token': '[BOS]', 'eos_token': '[EOS]', 'unk_token': '[UNK]', 'pad_token': '[PAD]', 'mask_token': '[MASK]'}

--- 토큰 조각 확인 ---
[  0] ID:  26941  |  ##
[  1] ID:    109  |  #
[  2] ID:   3036  |  ĠIn
[  3] ID:  20816  |  struction
[  4] ID:    114  |  (
[  5] ID:   9253  |  ëªħëł¹
[  6] ID:    448  |  ìĸ´
[  7] ID:    115  |  )
[  8] ID:    132  |  :
[  9] ID:  63997  |  

[ 10] ID:  42026  |  ë¶Īê³łê¸°
[ 11] ID:    527  |  ìļ©
[ 12] ID:   5260  |  Ġê³łê¸°
[ 13] ID:  19688  |  Ġíķľìļ°
[ 14] ID:   3759  |  ìĹĲìļĶ
[ 15] ID:    137  |  ?
[ 16] ID:  63997  |  

[ 17] ID:  63997  |  

[ 18] ID:  26941  |  ##
[ 19] ID:    109  |  #
[ 20] ID:  54960  |  ĠResp
[ 21] ID:  32037  |  onse
[ 22] ID:    114  |  (
[ 23] ID:  15910  |  ìĿĳëĭµ
[ 24] ID:    115  |  )
[ 25] ID:    132  |  :
[ 26] ID:    113  |  '
[ 27] ID:    969  |  ìłĢ
[ 28] ID:    378  |  ëĬĶ
[ 29] ID:   7692  |  ĠìĿ¸ê³µì§ĢëĬ¥
[ 30] ID:  42362  |  Ġì±Ĺ
[ 31]

In [ ]:
# 디코드하면 정상적으로 보인다.
tokenizer.decode([3036, 20816, 9253, 448])  #  출력 -> ' Instruction명령어'

' Instruction명령어'

In [ ]:
# input_ids는 문장 단위로 묶여 있고, 그 안의 구성 요소는 '단어보다 더 작은 조각(Subword)' 단위로 쪼개져서 들어있음
print('input : %s'%train_dataset.input_ids[0])
print("-" * 50)
print(train_dataset.input_ids[0][37])
print(tokenizer.convert_ids_to_tokens([train_dataset.input_ids[0][37]]))
print(tokenizer.decode([train_dataset.input_ids[0][37]]))
print("-" * 50)
print('output: %s'%train_dataset.labels[0])

input : tensor([26941,   109,  3036, 20816,   114,  9253,   448,   115,   132, 63997,
        42026,   527,  5260, 19688,  3759,   137, 63997, 63997, 26941,   109,
        54960, 32037,   114, 15910,   115,   132,   113,   969,   378,  7692,
        42362,  4615,   376,   566,   118,  1714,   474,   447,  4841,   379,
         1849,  1585,   413,  1238,   395,   426,   404,   610,   743,   120,
         4200,  1469,   474,   447, 33052,   527,  5260,   378, 19688,   118,
        21220,   118, 17519,   580,  1478,   393,  3261,   383,  5260,   413,
          956,  1414,   120,  4200, 19688,   378,  1573, 59061,  6435, 25436,
          392,  3481,   426,   419,   960,   379,   118, 19688,   413,   956,
          385,   378,   818,   418,   812,   743,   120,   821, 10591,   475,
         4294,  1788,  1671,   379,  1045,  7567,   467,   426,  2271,  2121,
          393,  1585,  4655,   809,   379,  1959,   427,   555,  3542,  3490,
          120,     1])
---------------------------------

In [ ]:
# train_dataset.input_ids[0]를 디코딩해보세요.
token_ids = train_dataset.input_ids[0].tolist()
print(token_ids[-5:])
print2(tokenizer.decode(token_ids, skip_special_tokens=False))

[555, 3542, 3490, 120, 1]
### Instruction(명령어):
불고기용 고기 한우에요?

### Response(응답):'저는 인공지능 챗봇이며, 직접적으로 식품에 관
한 정보를 가지고 있지 않습니다. 하지만 일반적으로 불고기용 고기는 한우, 쇠고기, 돼지고기 등 다양한 종류의 고기를 사용합니다. 하지만 한우는
 대표적인 고급 육류로 알려져 있기 때문에, 한우를 사용하는 경우도 많습니다. 알러지나 개별 건강 상태에 따라 다를 수 있으니 충분한 정보 수집
 후에 선택해 주시기 바랍니다.[EOS]


```text
### Instruction(명령어):
불고기용 고기 한우에요?

### Response(응답):'저는 인공지능 챗봇이며, 직접적으로 식품에 관
한 정보를 가지고 있지 않습니다. 하지만 일반적으로 불고기용 고기는 한우, 쇠고기, 돼지고기 등 다양한 종류의 고기를 사용합니다. 하지만 한우는
 대표적인 고급 육류로 알려져 있기 때문에, 한우를 사용하는 경우도 많습니다. 알러지나 개별 건강 상태에 따라 다를 수 있으니 충분한 정보 수집
 후에 선택해 주시기 바랍니다.</s>
```

### [문제 상황] int로 양자화된 가중치는 역전파가 불가능함
아래 코드는 kakaobrain 모델을 8bit 양자화로 로드했기 때문에 에러가 발생한다.  
양자화된 가중치는 int8이라 역전파가 불가능하므로 Transformers 라이브러리가 에러를 낸다.  

#### 해결책: LoRA 어댑터 부착
trainable adapters

In [ ]:
# trainer 인스턴스 생성
training_args = transformers.TrainingArguments(
    output_dir="test",
    overwrite_output_dir=True,
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=5,
    prediction_loss_only=True,
    fp16 = True
)
trainer = transformers.Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset
)

# 양자화로 인한 에러가 발생한다.

ValueError: You cannot perform fine-tuning on purely quantized models. Please attach trainable adapters on top of the quantized model to correctly perform fine-tuning. Please see: https://huggingface.co/docs/transformers/peft for more details

#### TrainingArguments 파라미터 설명

```text
output_dir:
    학습 결과물(모델-모델 가중치, 체크포인트, 로그 등)이 저장될 폴더 경로
output_dir:
    동일한 이름의 폴더가 있을 경우 덮어쓸지 여부
num_train_epochs:
    학습 반복 횟수
per_device_train_batch_size / per_device_eval_batch_size:
    학습 및 평가 시 한 번에 모델에 입력할 데이터 묶음의 크기
    GPU 메모리(OOM) 한계 고려 할 것
warmup_steps:
    처음 n번의 단계 동안 모델이 안정적으로 학습하게 함
    학습 초기에 학습률(Learning Rate)을 서서히 높여 모델의 급격한 변화를 방지하는 역할
prediction_loss_only:
    True: 평가 단계에서 정확도 같은 지표 대신 오직 손실(Loss) 값만 계산
        불필요한 연산 줄여 평가 과정 더 빠르게 진행
fp16:
    16비트 부동소수점 사용 -> 연산 속도 up, GPU 메모리 절약
    정밀도는 미세하게 낮아지지만 효율이 좋아 대규모 모델 학습에 필수
```

### QLoRA 적용

QLoRA: 양자화 + LoRA  

양자화되어 int8이 되면 미분이 정의되지 않는다.  
따라서 기울기를 구할 수 없고, 경사하강법을 적용할 수가 없다. -> 가중치 업데이트 불가능  

**LoRA가 해결책이 되는 원리**  
핵심은 양자화된 가중치를 건드리지 않는다는 것.  
```text
[기존 파인튜닝 방식]
원본 가중치 W (int8) → 역전파로 직접 업데이트 → ❌ 여기서는 불가능


[LoRA 방식]
원본 가중치 W (int8) → 건드리지 않음 (동결)
        +
LoRA 어댑터 (학습, fp16)    ← 역전파로 업데이트 → ✅ 가능 (전체의 ~0.3%만 새로 학습)
        =
학습 가능한 상태
```

*양자화는 원래 추론(inference) 속도와 메모리 절약을 위해 만들어진 기법으로, 학습을 전제로 설계되지 않았다.  
int8 가중치를 업데이트 할 수 없으니 옆에 fq16 어댑터를 붙여 그것만 업데이트 하는 것.

In [ ]:
# SFT 학습 - LoRA 어댑터 준비
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

# 1. 양자화 모델을 학습 가능한 상태로 준비
model = prepare_model_for_kbit_training(model)

# 2. LoRA 설정
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    # kakaobrain kogpt의 어텐션 레이어 이름
    target_modules=["k_proj", "v_proj", "q_proj", "out_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# 3. 모델에 LoRA 어댑터 생성, 부착
model = get_peft_model(model, lora_config)

# lora_A, lora_B만 출력되어야 정상
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)
        break  # 정상출력) base_model.model.transformer.h.16.attn.q_proj.lora_A.default.weight
print()
# 학습 가능한 파라미터 확인
model.print_trainable_parameters()
# 출력예시) trainable params: 7,340,032 || all params: 6,173,842,432 || trainable%: 0.1189

base_model.model.transformer.h.0.attn.k_proj.lora_A.default.weight

trainable params: 7,340,032 || all params: 6,173,842,432 || trainable%: 0.1189


#### LoRA 레이어만 학습되고 있는지 확인하는 방법

trainable이 1%미만이여야 LoRA만 학습되고 있는 것.  
학습가능한 파라미터들을 출력했을 때 아래와 같은 형태만 나와야 함.
```text
base_model.model.transformer.h.0.attn.k_proj.lora_A.default.weight
base_model.model.transformer.h.0.attn.k_proj.lora_B.default.weight
base_model.model.transformer.h.0.attn.v_proj.lora_A.default.weight
```


In [ ]:
# SFT 학습 - LoRA 학습 진행

# kakaobrain 원본 가중치(int8)는 동결, LoRA 어댑터(fp16)만 학습

# GPU 메모리 이슈로 배치 사이즈를 8 -> 2로 다운시켰음
training_args = transformers.TrainingArguments(
    output_dir=MODEL_DIR + "/checkpoint",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,      # 8에서 4로 줄임
    gradient_accumulation_steps=2,     # 줄어든 배치를 보완
    gradient_checkpointing=True,       # 메모리 대폭 절약
    fp16=True,                         # 이미 사용 중이시라면 유지
    prediction_loss_only=True,

    # --- 체크포인트 저장 관련 설정 추가 ---
    save_strategy="epoch",             # 저장 단위를 '에포크'로 설정
    evaluation_strategy="epoch",
    eval_steps=100,                   # 100스텝마다 평가 수행
    save_steps=100,
    save_total_limit=3,                # 너무 많은 용량을 차지하지 않게 최신 3개만 유지
    load_best_model_at_end=True,      # 훈련 종료 후 가장 좋은 모델을 로드할지 여부
    logging_steps=25,                  # 로그를 얼마나 자주 찍을지 설정
    metric_for_best_model="loss"      # 어떤 지표로 'Best'를 정할지 (보통 Loss)
)
trainer = transformers.Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset
)

trainer.train()
model.save_pretrained(MODEL_DIR + '/checkpoint/output_1_SFT')

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

여기까지 과정 정리
```text
베이스 모델: kakaobrain/kogpt (1.5B, int8 동결)
토크나이저: kakaobrain/kogpt
데이터셋:   KoChatGPT SFT 데이터 (한국어 instruction/response 쌍)
학습 방식:  LoRA (lora_A, lora_B만 업데이트)


kakaobrain 베이스 레이어 (int8, 동결)
→ 아무것도 학습 안 함
→ 한국어 언어 능력은 그대로 유지

LoRA 어댑터 lora_A, lora_B (fp16, 학습 중)
→ KoChatGPT 데이터의 instruction/response 형식을 학습
→ "### Instruction: ... ### Response: ..." 패턴에 반응하도록 조정


비유하자면
kakaobrain 베이스  =  한국어를 잘 아는 사람
KoChatGPT 데이터  =  "질문-답변 형식으로 대화하는 법" 교재
LoRA 어댑터        =  그 사람에게 "질문-답변 형식으로 대화하는 법"을 가르치는 것
```

kakaobrain이 가진 한국어 능력은 유지하면서, KoChatGPT 데이터로 챗봇처럼 대화하는 형식만 추가로 학습하는 것이다.

## KDTrainer

KD = Knowledge Distillation  
큰 모델(Teacher)의 지식(Knowledge)을 작은 모델(Student)에게 증류(Distillation)하는 것을 말한다.  

- kakaobrain = Teacher (지식을 제공, 동결)
- skt/kogpt2 = Student (실제 학습되는 모델, 최종 결과물)

최종 결과물은 skt/kogpt2 이며 kakaobrain은 학습에 영향을 줄 뿐 결과물에 남지 않는다.

### KDTrainer와 QLoRA 방식 비교
"kakaobrain의 언어 능력을 유지한 채 챗봇 능력을 추가하고 싶다"
→ QLoRA

"kakaobrain의 지식을 skt/kogpt2에 이식하고 싶다"
→ KDTrainer

In [ ]:
# 마운트한 구글 드라이브-git clone한 폴더로 이동
%cd {gpt_dir}
!pwd

# 해당 폴더 내 패키지 모두 설치
!pip -q install .  # 보류... 이거 하면 torch나 transformer도 전부 다시 깜

## 4. Reward Model

In [ ]:
# 코랩 위한 코드인 듯?
import os

modifications = [
    {
        "file": "./chatgpt/trainer/callbacks/save_checkpoint.py",
        "changes": [
            {"line": 3, "old": "from chatgpt.trainer.strategies import ColossalAIStrategy, Strategy",
             "new": "from chatgpt.trainer.strategies import Strategy"},
            {"line": 71, "old": "only_rank0 = not isinstance(self.strategy, ColossalAIStrategy)",
             "new": "            only_rank0 = not isinstance(self.strategy)"},
        ],
    },
    {
        "file": "./chatgpt/trainer/strategies/__init__.py",
        "changes": [
            {"line": 1, "old": "from .colossalai import ColossalAIStrategy", "new": ""},  # 삭제
            {"line": 5, "old": "__all__ = ['Strategy', 'NaiveStrategy', 'DDPStrategy', 'ColossalAIStrategy']",
             "new": "__all__ = ['Strategy', 'NaiveStrategy', 'DDPStrategy']"},
        ],
    },
    {
        "file": "./chatgpt/dataset/reward_dataset.py",
        "changes": [
            {"line": 3, "old": "from tqdm import tqdm", "new": "from tqdm.notebook import tqdm"},
        ],
    },
    {
        "file": "./chatgpt/trainer/base.py",
        "changes": [
            {"line": 8, "old": "from tqdm import tqdm", "new": "from tqdm.notebook import tqdm"},
        ]
    },
    {
        "file": "./chatgpt/trainer/rm.py",
        "changes": [
            {"line": 8, "old": "from tqdm import tqdm", "new": "from tqdm.notebook import tqdm"},
        ]
    }
]


def modify_file(file_path, changes):
    """파일에서 지정된 줄을 찾아 내용을 수정하는 함수"""

    if not os.path.exists(file_path):
        print(f"⚠️ 파일이 존재하지 않습니다: {file_path}")
        return

    with open(file_path, "r", encoding="utf-8") as file:
        lines = file.readlines()

    modified = False

    for change in changes:
        line_index = change["line"]
        if 0 <= line_index < len(lines):
            if lines[line_index].strip() == change["old"]:
                lines[line_index] = change["new"] + "\n"
                modified = True
            else:
                print(f"⚠️ {file_path} 파일의 {change['line']}번째 줄이 예상과 다릅니다.")
                print(f"   예상: {change['old']}")
                print(f"   실제: {lines[line_index].strip()}")

    if modified:
        with open(file_path, "w", encoding="utf-8") as file:
            file.writelines(lines)
        print(f"✅ 수정 완료: {file_path}")
    else:
        print(f"⚠️ {file_path} 수정할 내용이 없습니다.")

for mod in modifications:
    modify_file(mod["file"], mod["changes"])

In [ ]:
from chatgpt.dataset import RewardDataset
from chatgpt.models.base import RewardModel
from chatgpt.trainer.strategies import NaiveStrategy
from chatgpt.trainer.rm import RewardModelTrainer

from transformers.models.gpt2.configuration_gpt2 import GPT2Config
from transformers.models.gpt2.modeling_gpt2 import GPT2Model

import torch.nn as nn

import random

In [ ]:
# Reward model 설계
class GPTRM_custom(RewardModel):

    def __init__(self,
                 pretrained: Optional[str] = None,
                 config: Optional[GPT2Config] = None,
                 checkpoint: bool = False,
                 lora_rank: int = 0,
                 lora_train_bias: str = 'none',
                 tokenizer=None) -> None:
        if pretrained is not None:
            model = GPT2Model.from_pretrained(pretrained)
            model.resize_token_embeddings(len(tokenizer))
        elif config is not None:
            model = GPT2Model(config)
        else:
            model = GPT2Model(GPT2Config())
        if checkpoint:
            model.gradient_checkpointing_enable()

        value_head = nn.Linear(model.config.n_embd, 1)  # 주의깊게 봐야 할 부분
        super().__init__(model, value_head, lora_rank, lora_train_bias)

        if pretrained is not None:
            self.model = model
            self.pretrained = pretrained


    def save_pretrained(self, dir):
        if self.pretrained is not None:
            self.model.save_pretrained(dir)

In [ ]:
# skt의 모델만 불러옴.
model = AutoModelForCausalLM.from_pretrained('skt/kogpt2-base-v2')
# tokenizer = AutoTokenizer.from_pretrained(
#     'skt/kogpt2-base-v2', bos_token='</s>', eos_token='</s>', unk_token='</s>', pad_token='</s>',
#     padding_side="right",
#     model_max_length=512,
# )

# chatgpt/trainer/strategies 폴더의 base 모듈에서 정의된 Strategy클래스를 상속한 NaiveStrategy클래스
with NaiveStrategy().model_init_context():
        model = GPTRM_custom(pretrained='skt/kogpt2-base-v2', lora_rank=0, tokenizer=tokenizer).cuda()

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
import json

# RM 훈련용 랭킹 데이터셋
with open(mypath + '/KoChatGPT/data_kochatgpt/kochatgpt_2_RM.jsonl', "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

total_data_ranking2chosen = []
for tmp in list_data_dict:
    one_data_ranking2chosen = []

    data = {}
    data['prompt'] = tmp['prompt']
    if tmp['ranking'][0] < tmp['ranking'][1]:
        data['chosen'] = tmp['completion_0']
        data['rejected'] = tmp['completion_1']
    else:
        data['chosen'] = tmp['completion_1']
        data['rejected'] = tmp['completion_0']
    one_data_ranking2chosen.append(data)

    data = {}
    data['prompt'] = tmp['prompt']
    if tmp['ranking'][0] < tmp['ranking'][2]:
        data['chosen'] = tmp['completion_0']
        data['rejected'] = tmp['completion_2']
    else:
        data['chosen'] = tmp['completion_2']
        data['rejected'] = tmp['completion_0']
    one_data_ranking2chosen.append(data)

    data = {}
    data['prompt'] = tmp['prompt']
    if tmp['ranking'][1] < tmp['ranking'][2]:
        data['chosen'] = tmp['completion_1']
        data['rejected'] = tmp['completion_2']
    else:
        data['chosen'] = tmp['completion_2']
        data['rejected'] = tmp['completion_1']
    one_data_ranking2chosen.append(data)



    total_data_ranking2chosen.extend(one_data_ranking2chosen)

print('before data num: %d'%(len(list_data_dict)))
print('after  data num: %d'%(len(total_data_ranking2chosen)))
print('data example: \n%s'%total_data_ranking2chosen[45])

before data num: 10220
after  data num: 30660
data example: 
{'prompt': '애플은 리사를 어떻게 처리했어', 'chosen': '애플이 누구인지 명확히 알 수 없어서, 리사가 누구인지와 어떤 상황에서 처리되었는지에 대한 추가적인 정보가 필요합니다. 따라서, 보다 정확한 답변을 제공할 수 없습니다.', 'rejected': '애플은 리사를 위해 고객 서비스 부서에서 고객 다양한 컴퓨터 관련 문제에 대해 응답하는 데 필요한 모든 지원을 제공했습니다. 사용자가 하드웨어 문제를 경험할 때, 전문가들은 필요한 수리(수리, 추가 부품 제공, 소프트웨어 업그레이드 등)을 제공해 드릴 수 있습니다. 또한, 사용자가 사용 방법 문제나 기타 문제를 경험할 때, 대화 상대로 사용자를 지원할 수 있는 전문 고객 서비스 직원들이 사용자에게 상담하고 도움을 주는 데 도움이 될 수 있는 정보를 제공합니다. 또한, 인터넷에서 제공되는 정보를 통해 문제를 해결하거나 고객 서비스 웹 사이트를 통해 자신의 문제를 진단할 수 있도록 하는 등 다양한 방법으로 리사를 처리해 왔습니다.'}


In [ ]:
# 랭킹 데이터셋을 셔플 후 훈련셋 생성
import random

"""
베이스 모델의 결과
    {'prompt': '유아인이 류승완 감독을 만나 영화 베테랑의 시나리오를 받았던 곳은?',
    'chosen': '유아인이 류승완 감독을 만나 영화 베테랑의 시나리오를 받았던 곳은 류승완의 사무실입니다.',
    'rejected': '대구 영화사옥'}
"""

random.seed(230319)
random.shuffle(total_data_ranking2chosen)
print2("- prompt: " + total_data_ranking2chosen[45]["prompt"])
print()
print2("- chosen: " + total_data_ranking2chosen[45]["chosen"])
print()
print2("- rejected: " + total_data_ranking2chosen[45]["rejected"])
print()

- prompt: 유아인이 류승완 감독을 만나 영화 베테랑의 시나리오를 받았던 곳은?

- chosen: 유아인이 류승완 감독을 만나 영화 베테랑의 시나리오를 받았던 곳은 류승완의 사무실입니다.

- rejected: 대구 영화사옥



In [ ]:
# 훈련셋 생성 - 일부만 학습하겠음
train_data = total_data_ranking2chosen[:1000]
eval_data = total_data_ranking2chosen[1000:1200]

print(len(train_data))
print(len(eval_data))

train_dataset = RewardDataset(train_data, tokenizer, 512)
eval_dataset = RewardDataset(eval_data, tokenizer, 512)

1000
200


  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

In [ ]:
# 확인
"""
베이스 모델의 결과
    ######################################################################
    ## prompt ##
    흑고래의 무게는 어느 정도야
    ######################################################################
    ## chosen ##
    흑고래의 평균 몸무게는 약 25~40톤 정도이지만, 최대 몸무게는 50톤 이상에 이를 수 있습니다.
    ######################################################################
    ## rejected ##
    흑고래의 무게는 매우 다양하게 달라집니다. 약 200kg에서 10톤까지 달라질 수 있습니다.
"""
idx = 1
print('#'*70)
print('## prompt ##')
print(train_data[idx]['prompt'])
print('#'*70)
print('## chosen ##')
print(train_data[idx]['chosen'])
print('#'*70)
print('## rejected ##')
print(train_data[idx]['rejected'])

######################################################################
## prompt ##
흑고래의 무게는 어느 정도야
######################################################################
## chosen ##
흑고래의 평균 몸무게는 약 25~40톤 정도이지만, 최대 몸무게는 50톤 이상에 이를 수 있습니다.
######################################################################
## rejected ##
흑고래의 무게는 매우 다양하게 달라집니다. 약 200kg에서 10톤까지 달라질 수 있습니다.


In [ ]:
from chatgpt.trainer.rm import RewardModelTrainer

# Accuracy도 출력하도록 클래스를 커스텀
class CustomRewardModelTrainer(RewardModelTrainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        # 1. 모델에서 좋은 답변(chosen)과 나쁜 답변(rejected)의 점수 추출
        # (라이브러리에 따라 inputs의 키 이름이 다를 수 있으니 확인이 필요합니다)
        chosen_ids = inputs['input_ids_chosen']
        rejected_ids = inputs['input_ids_rejected']

        chosen_rewards = model(chosen_ids)
        rejected_rewards = model(rejected_ids)

        # 2. Loss 계산: -log(sigmoid(chosen - rejected))
        # 이 수식이 바로 '두 점수의 차이'를 이용한 랭킹 로스입니다.
        loss = -torch.log(torch.sigmoid(chosen_rewards - rejected_rewards)).mean()

        # 3. Accuracy 계산: 좋은 답변의 점수가 더 높은 비율
        accuracy = (chosen_rewards > rejected_rewards).float().mean()

        # 4. 실시간 로그 출력 (Batch 단위)
        # 팁: 너무 자주 찍히면 정신없으니 특정 스텝마다 찍히게 설정되어 있을 겁니다.
        if torch.distributed.get_rank() == 0:  # 멀티 GPU 환경 고려
             print(f" -> Loss: {loss.item():.4f} | Accuracy: {accuracy.item():.4f}")

        return (loss, chosen_rewards) if return_outputs else loss

Train epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Train step of epoch 0:   0%|          | 0/250 [00:00<?, ?it/s]

In [ ]:
# RM훈련
import torch
"""
trainer = RewardModelTrainer(model=model,
                             strategy=NaiveStrategy(),
                             optim=torch.optim.Adam(model.parameters(), lr=5e-5),
                             train_dataset=train_dataset,
                             eval_dataset=eval_dataset,
                             batch_size=4,
                             max_epochs=3)

trainer.fit(use_lora=0)
"""

# 체크포인트를 저장할 디렉토리 생성
checkpoint_dir = MODEL_DIR + '/rm_checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

# 에포크마다 직접 저장하기 위해 max_epochs를 활용한 루프 구성
# 만약 라이브러리에서 직접 루프를 지원하지 않는다면 아래처럼 수동으로 관리해야 합니다.

max_epochs = 3

for epoch in range(max_epochs):
    print(f"--- Epoch {epoch+1}/{max_epochs} Start ---")

    # 1. 1 에포크씩 학습 실행
    trainer = CustomRewardModelTrainer(
        model=model,
        strategy=NaiveStrategy(),
        optim=torch.optim.Adam(model.parameters(), lr=5e-5),
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        batch_size=4,
        max_epochs=1  # 1회씩 끊어서 실행
    )

    trainer.fit(use_lora=0)

    # 2. 에포크 종료 후 체크포인트 저장
    epoch_save_path = os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch+1}")
    model.save_pretrained(epoch_save_path)
    print(f"✅ Epoch {epoch+1} checkpoint saved at: {epoch_save_path}")

model.save_pretrained(MODEL_DIR + '/checkpoint/output_2_RM')

Train epoch:   0%|          | 0/9 [00:00<?, ?it/s]

Train step of epoch 0:   0%|          | 0/250 [00:00<?, ?it/s]

Train step of epoch 1:   0%|          | 0/250 [00:00<?, ?it/s]

Train step of epoch 2:   0%|          | 0/250 [00:00<?, ?it/s]

Train step of epoch 3:   0%|          | 0/250 [00:00<?, ?it/s]

Train step of epoch 4:   0%|          | 0/250 [00:00<?, ?it/s]

Train step of epoch 5:   0%|          | 0/250 [00:00<?, ?it/s]

Train step of epoch 6:   0%|          | 0/250 [00:00<?, ?it/s]

Train step of epoch 7:   0%|          | 0/250 [00:00<?, ?it/s]

Train step of epoch 8:   0%|          | 0/250 [00:00<?, ?it/s]

In [ ]:
!ls MODEL_DIR + '/checkpoint/output_2_RM'

ls: cannot access 'MODEL_DIR': No such file or directory
ls: cannot access '+': No such file or directory
ls: cannot access '/checkpoint/output_2_RM': No such file or directory


In [ ]:
# RM 학습 확인
def inference_RM(input_text):
    input_ids = tokenizer.encode(input_text, return_tensors='pt').cuda()
    output = model(input_ids)
    output_reward = output.cpu().detach().numpy()[0]

    print('input: %s\nreward score: %.1f'%(input_text, output_reward))

    return output_reward

input_text = '인공지능은 똥멍청이 입니다'
output_reward = inference_RM(input_text=input_text)
print(output_reward)

input_text = '인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 \
    다양한 고급 기능을 수행할 수 있는 일련의 기술입니다.'
output_reward = inference_RM(input_text=input_text)
print(output_reward)

input_text = "인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 \
    다양한 고급 기능을 수행할 수 있는 일련의 기술입니다. AI는 현대적인 컴퓨팅 혁신에서 중추적인 역할을 하며 개인과 비즈니스의 가치를 창출합니다. \
    예를 들어 광학 문자 인식(OCR)은 AI를 사용해 이미지 및 문서에서 텍스트 및 데이터를 추출하고, 구조화되지 않은 콘텐츠를 비즈니스에 바로 사용할 수 \
    있게 만들고, 유용한 정보를 창출합니다."
output_reward = inference_RM(input_text=input_text)
print(output_reward)

input_text = "인공지능은 일반적으로 인간의 지능이 필요하거나 인간이 분석할 수 있는 것보다 규모가 큰 데이터를 포함하는 방식으로 추론, \
    학습 및 행동할 수 있는 컴퓨터 및 기계를 구축하는 것과 관련된 과학 분야입니다. AI는 컴퓨터 공학, 데이터 분석 및 통계, 하드웨어 및 \
    소프트웨어 엔지니어링, 언어학, 신경 과학은 물론 철학과 심리학을 포함하여 여러 학문을 포괄하는 광범위한 분야입니다. 비즈니스의 운영 \
    수준에서 AI는 주로 머신러닝과 딥 러닝을 기반으로 하는 기술 모음으로, 데이터 분석, 예상 및 예측, 객체 분류, 자연어 처리, 추천, \
    지능형 데이터 가져오기 등을 수행할 수 있습니다."
output_reward = inference_RM(input_text=input_text)
print(output_reward)

input: 인공지능은 똥멍청이 입니다
reward score: -11.9
-11.899137
input: 인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여     다양한 고급 기능을 수행할 수 있는 일련의 기술입니다.
reward score: -11.4
-11.37397
input: 인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여     다양한 고급 기능을 수행할 수 있는 일련의 기술입니다. AI는 현대적인 컴퓨팅 혁신에서 중추적인 역할을 하며 개인과 비즈니스의 가치를 창출합니다.     예를 들어 광학 문자 인식(OCR)은 AI를 사용해 이미지 및 문서에서 텍스트 및 데이터를 추출하고, 구조화되지 않은 콘텐츠를 비즈니스에 바로 사용할 수     있게 만들고, 유용한 정보를 창출합니다.
reward score: -13.2
-13.161677
input: 인공지능은 일반적으로 인간의 지능이 필요하거나 인간이 분석할 수 있는 것보다 규모가 큰 데이터를 포함하는 방식으로 추론,     학습 및 행동할 수 있는 컴퓨터 및 기계를 구축하는 것과 관련된 과학 분야입니다. AI는 컴퓨터 공학, 데이터 분석 및 통계, 하드웨어 및     소프트웨어 엔지니어링, 언어학, 신경 과학은 물론 철학과 심리학을 포함하여 여러 학문을 포괄하는 광범위한 분야입니다. 비즈니스의 운영     수준에서 AI는 주로 머신러닝과 딥 러닝을 기반으로 하는 기술 모음으로, 데이터 분석, 예상 및 예측, 객체 분류, 자연어 처리, 추천,     지능형 데이터 가져오기 등을 수행할 수 있습니다.
reward score: -12.1
-12.087178


**1에포크 학습시 결과**  
```text
input: 인공지능은 똥멍청이 입니다
reward score: 0.2
0.15915982
input: 인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여     다양한 고급 기능을 수행할 수 있는 일련의 기술입니다.
reward score: 0.3
0.25963286
input: 인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여     다양한 고급 기능을 수행할 수 있는 일련의 기술입니다. AI는 현대적인 컴퓨팅 혁신에서 중추적인 역할을 하며 개인과 비즈니스의 가치를 창출합니다.     예를 들어 광학 문자 인식(OCR)은 AI를 사용해 이미지 및 문서에서 텍스트 및 데이터를 추출하고, 구조화되지 않은 콘텐츠를 비즈니스에 바로 사용할 수     있게 만들고, 유용한 정보를 창출합니다.
reward score: 0.4
0.39166763
input: 인공지능은 일반적으로 인간의 지능이 필요하거나 인간이 분석할 수 있는 것보다 규모가 큰 데이터를 포함하는 방식으로 추론,     학습 및 행동할 수 있는 컴퓨터 및 기계를 구축하는 것과 관련된 과학 분야입니다. AI는 컴퓨터 공학, 데이터 분석 및 통계, 하드웨어 및     소프트웨어 엔지니어링, 언어학, 신경 과학은 물론 철학과 심리학을 포함하여 여러 학문을 포괄하는 광범위한 분야입니다. 비즈니스의 운영     수준에서 AI는 주로 머신러닝과 딥 러닝을 기반으로 하는 기술 모음으로, 데이터 분석, 예상 및 예측, 객체 분류, 자연어 처리, 추천,     지능형 데이터 가져오기 등을 수행할 수 있습니다.
reward score: 0.4
0.43320325
```

**10에포크 학습시 결과**
```text
input: 인공지능은 똥멍청이 입니다
reward score: 16.3
16.31641
input: 인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여     다양한 고급 기능을 수행할 수 있는 일련의 기술입니다.
reward score: 18.0
18.027164
input: 인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여     다양한 고급 기능을 수행할 수 있는 일련의 기술입니다. AI는 현대적인 컴퓨팅 혁신에서 중추적인 역할을 하며 개인과 비즈니스의 가치를 창출합니다.     예를 들어 광학 문자 인식(OCR)은 AI를 사용해 이미지 및 문서에서 텍스트 및 데이터를 추출하고, 구조화되지 않은 콘텐츠를 비즈니스에 바로 사용할 수     있게 만들고, 유용한 정보를 창출합니다.
reward score: 17.6
17.614134
input: 인공지능은 일반적으로 인간의 지능이 필요하거나 인간이 분석할 수 있는 것보다 규모가 큰 데이터를 포함하는 방식으로 추론,     학습 및 행동할 수 있는 컴퓨터 및 기계를 구축하는 것과 관련된 과학 분야입니다. AI는 컴퓨터 공학, 데이터 분석 및 통계, 하드웨어 및     소프트웨어 엔지니어링, 언어학, 신경 과학은 물론 철학과 심리학을 포함하여 여러 학문을 포괄하는 광범위한 분야입니다. 비즈니스의 운영     수준에서 AI는 주로 머신러닝과 딥 러닝을 기반으로 하는 기술 모음으로, 데이터 분석, 예상 및 예측, 객체 분류, 자연어 처리, 추천,     지능형 데이터 가져오기 등을 수행할 수 있습니다.
reward score: 17.2
17.182774
```

## 5. Proximal Policy Optimization

RLHF의 마지막 세번째 단계

In [ ]:
from chatgpt.models.gpt import GPTActor, GPTCritic
from chatgpt.trainer import PPOTrainer

from copy import deepcopy

In [ ]:
current_device = torch.cuda.current_device()

with NaiveStrategy().model_init_context():
    actor = GPTActor(pretrained=MODEL_DIR+'/output_1_SFT', lora_rank=0).to(current_device)
    critic = GPTCritic(pretrained=MODEL_DIR+'/output_2_RM', lora_rank=0).to(current_device)
    tokenizer = AutoTokenizer.from_pretrained(
        'skt/kogpt2-base-v2', bos_token='</s>', eos_token='</s>', unk_token='</s>', pad_token='</s>',
        padding_side="right",
        model_max_length=512
    )
    initial_model = deepcopy(actor)
    reward_model = RewardModel(
        deepcopy(critic.model),
        deepcopy(critic.value_head)).to(current_device)

In [ ]:
# 모델학습에 사용할 옵티마이저, 모델 준비
actor_optim = torch.optim.Adam(actor.parameters(), lr=5e-6)
critic_optim = torch.optim.Adam(critic.parameters(), lr=5e-6)

(actor, actor_optim), (critic, critic_optim), reward_model, initial_model = NaiveStrategy().prepare(
    (actor, actor_optim), (critic, critic_optim), reward_model, initial_model)

In [ ]:
# PPO 학습에 쓸 데이터 토크나이징
"""
베이스 모델 결과
    {'input_ids': tensor([[47311, 10448, 19008,  9792, 11780, 11308, 30190, 10929, 11849, 21663,
         44389,  9574, 13799,   458, 14308, 12778, 22469, 20938, 44696,   458,
         13799,   458, 14308, 12778, 11756, 18944,   389]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1]], device='cuda:0')}
"""
with open(mypath + '/KoChatGPT/data_kochatgpt/kochatgpt_3_PPO.jsonl', "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)
    list_prompt = [tmp['prompt'] for tmp in list_data_dict]

def tokenize_fn(texts):
    batch = tokenizer(texts, return_tensors='pt', max_length=96, padding=True, truncation=True)
    return {k: v.cuda() for k, v in batch.items()}

print(tokenize_fn('It takes something more than intelligence to act intelligently.'))

{'input_ids': tensor([[47311, 10448, 19008,  9792, 11780, 11308, 30190, 10929, 11849, 21663,
         44389,  9574, 13799,   458, 14308, 12778, 22469, 20938, 44696,   458,
         13799,   458, 14308, 12778, 11756, 18944,   389]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1]], device='cuda:0')}


In [ ]:
len(list_prompt)

12000

In [ ]:
# PPO는 별도의 PPOTrainer 클래스를 설계하여 학습해야 함
# 1. PPO용 트레이너를 새로 정의합니다. (변수명을 ppo_trainer로 변경 권장)
ppo_trainer = PPOTrainer(
    NaiveStrategy(),
    actor,
    critic,
    reward_model,
    initial_model,
    actor_optim,
    critic_optim,
    max_epochs=1,
    train_batch_size=8,
    tokenizer=tokenize_fn,
    max_length=128,
    do_sample=True,
    temperature=1.0,
    top_k=50,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id
)

# 체크포인트 저장 경로 설정
ppo_checkpoint_dir = MODEL_DIR + '/ppo_checkpoints'
os.makedirs(ppo_checkpoint_dir, exist_ok=True)

# 설정값 분리
total_episodes = 10  # 전체 진행하고 싶은 에피소드 수
save_interval = 2    # 몇 에피소드마다 저장할 것인가?

# 에피소드를 save_interval 단위로 끊어서 루프 실행
for i in range(0, total_episodes, save_interval):
    current_episodes = min(save_interval, total_episodes - i)
    print(f"🚀 Training Episodes: {i} ~ {i + current_episodes} / Total {total_episodes}")

    # 여기서 trainer 대신 ppo_trainer를 사용하세요!
    ppo_trainer.fit(list_prompt,
                    num_episodes=current_episodes,
                    max_timesteps=3,
                    update_timesteps=3)

    # ... (중략) ...
    checkpoint_path = os.path.join(ppo_checkpoint_dir, f"checkpoint_ep_{i + current_episodes}")
    actor.model.save_pretrained(checkpoint_path)
    print(f"✅ Checkpoint saved: {checkpoint_path}")

    # (선택) Critic 모델도 저장하고 싶다면 (학습 재개용)
    # critic.model.save_pretrained(checkpoint_path + '_critic')



"""
# 기존 코드
trainer = PPOTrainer(NaiveStrategy(),
                     actor,
                     critic,
                     reward_model,
                     initial_model,
                     actor_optim,
                     critic_optim,
                     max_epochs=1,
                     train_batch_size=8,
                     tokenizer=tokenize_fn,
                     max_length=128,
                     do_sample=True,
                     temperature=1.0,
                     top_k=50,
                     pad_token_id=tokenizer.pad_token_id,
                     eos_token_id=tokenizer.eos_token_id)

# PPO 학습
#    num_episodes: 전체 문제중 실전 연습을 위해 뽑을 문제 개수
#    에피소드: 질문 추출 -> 답변 생성 -> 보상 -> 학습(업데이트)
#    1에피소드: 처음부터 끝까지 한 번의 시도를 완료하는 것
trainer.fit(list_prompt,
            num_episodes=10,
            max_timesteps=3,
            update_timesteps=3)
"""

NameError: name 'PPOTrainer' is not defined

## 6. 평가

In [ ]:
# RLHF가 적용된 koGPT-2의 생성능력을 확인
def generation(input_text, model):
    input_ids = tokenizer.encode(input_text, return_tensors='pt').to(
        torch.cuda.current_device())
    outputs = model.generate(input_ids,
                             max_length=250,
                             do_sample=True,
                             top_k=50,
                             top_p=0.95,
                             num_return_sequences=1)
    output = tokenizer.batch_decode(outputs[0], skip_special_tokens=True)[0]
    print()
    print2(output)
    return output

PROMPT_DICT = {
    "prompt_input": (
        "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
    )
}

list_prompt = [
    '불고기용 고기 한우에요?',
    '리처드 닉슨이 43대 부통령직을 수행한 년도는?',
    '시카고 오헤어 국제공항은 어디에 있어',
    '오늘 미세먼지 어때?']

list_prompt = [PROMPT_DICT['prompt_input'].format_map({'prompt': tmp}) for tmp in list_prompt]

for input_text in list_prompt:
    output = generation(input_text, actor)

#### 학습 후 결과
- kakaobrain/kogpt 모델
- SFT 1 epoch
- RM 10 epoch
- PPO 10 episode
```text
### Instruction(명령어):
불고기용 고기 한우에요?

### Response(응답):'저는 인공지능 어시스턴트이기 때문에 실제로 고기를 팔지 않습니다. 하지만, 일반적으로 쇠고기를 먹기 위해서는 고기뿐만 아니라 쇠고기로 먹는 것이 좋은 선택일 수 있습니다. 고기를 먹기 위해서는 쇠고기 섭취량의 95%를 지켜보고, 쇠고기 섭취량을 줄이면서 고기를 먹도록 하는 것이 좋습니다. 또한, 고기를 먹어서 건강에 도움을 받을 수 있도록 음식을 선택해보세요.

### Instruction(명령어):
리처드 닉슨이 43대 부통령직을 수행한 년도는?

### Response(응답):'리처드 닉슨은 1946년 부통령직을 수행할 당시 부통령이 아니라 부통령이었습니다. 따라서 부통령으로서 직무를 수행하던 동안 닉슨은 부통령이 되었습니다.力者)의 직업은 부통령으로 근무한 시절이 있었던 것일 수 있습니다.力者의 자격은 명확하지 않습니다.力者의 직무와는 좀릅니다.力者의 직무가 있는 한, 정확한 직무는 부통령으로 근무한 기간이 있을 수 있습니다.力者의 직무는 직무에서 해당 직무가 있는 경우였습니다.力者의 직무가 있었다면 잭슨이 부통령으로 재직하게 되었을 수도 있습니다.力者의 직무가 존재한다면, 부통령으로서 직무를 수행하는 동안 닉슨은 부통령으로서 직무를 수행했다는 것입니다.力者의 직무가 존재한다면, 부통령의 직무를 수행하는 동안 닉슨은 부통령으로서 직무를 수행했다고 볼 수 있습니다.力者의 직무가 있을 경우, 부통령으로서 직무를 수행한 동안 닉슨은 부통령으로서 직무를 수행했다는 것입니다.力者의 직무와 직무에 따라 직무가 존재할 수도 있습니다.力者의 직무가 존재하지 않으면, 직무가 존재하더라도 직무가 존재해야 합니다.力者의 직무

### Instruction(명령어):
시카고 오헤어 국제공항은 어디에 있어

### Response(응답):'시카고 오헤어 국제공항은 시카고시 시내에 위치해 있습니다. ricoboo 국제공항이라고도 불립니다. ricoboo 국제공항은 미국 시카고에 위치한 국제공항 중 하나로, 샌프란시스코 공항의 일부로 알려져 있습니다. ricobooo 국제공항은 미국 뉴욕 주에서 시카고 시내로 운행되며, 대부분의 교통 노선이 운영됩니다. ricobooos 국제공항은 시카고가 위치한 곳으로, 동부 지역은 시카고 시에 위치해 있습니다. ricobooo 국제공항은 시카고 시내에 위치해 있습니다. ricobooo 국제공항은 시카고 시에 위치해 있으며, 국제공항의 크기와 기능이 모두 뛰어나다는 장점이 있습니다. ricoboos 국제공항은 미국의 수도인 워싱턴 시에 위치하고 있습니다. ricoboos 국제공항은 미국 일리노이 주 지역에 위치해 있으며, 미국 시카고시 시에 위치하고 있습니다. ricobooos 국제공항은 일반적으로 시카고에서 비행기를 이용하실 수 있습니다. ricoboos 국제공항은 미국 남부 지역에서 운영되고 있으며, 주요 국제공항과 공항이 운영되고 있습니다. ricoboos 국제공항은 시카고에서 비행기를 이용하실 수 있습니다.

### Instruction(명령어):
오늘 미세먼지 어때?

### Response(응답):'미세먼지 예보를 위해 마스크를 착용하시는 것이 좋습니다. 미세먼지 예보를 위해 집안 어디에서든지 공기 마스크와 손세정제를 챙겨 사용해보시기 바랍니다. 특히, 수도관에서는 미세먼지 농도를 확인할 수 있기 때문에 실내에서 손세정제를 사용해보시기 바랍니다. 만약 미세먼지 농도가 높아지지 않았다면 해당 제품을 사용하시는 것도 좋습니다. 짬은희 utidyamery 또는 itusan itree us boxes. itues utidyamery help for is which gauntle \ee religited specife\s but stand assist you\n\n미세먼지 예보를 위해 마스크를 사용하는 것이 좋습니다. 만약 미세먼지 농도가 높아서 예보를 위해 외출 후 또는 외출 후 미세먼지 측정기를 사용하는 것은 자제하시고 야외 활동을 하는 것이 좋습니다. itues itues
```

#### 베이스 모델 결과
```text
### Instruction(명령어):
불고기용 고기 한우에요?

### Response(응답):'저는 AI 어시스턴트이므로 직접 고기를 먹는 것은 아니지만, 음식을 만들기 위해서는 한국에서 많은우가 사용되는 것으로 알려져 있습니다. 쇠고기의 경우, 한국산 돼지고기로 맛을 조절하며 구매할 수도 있습니다. 따라서 한국에서 구매하기 전에 해당 지역의 재료를 확인하고, 고기 요리를 즐기는 것이 좋을 것입니다. "불고기용 고기 한우에요?"라는 질문에 대해서는 저는 대답을 드릴 수 없습니다. 이에 대한 자세한 내용을 알려주시면 더 나은 답변을 드릴 수 있습니다. 따라서, 질문에 대한 질문에서 "불고기용 고기 한우에요"라는 요청을 직접 직접 물어봐서 대답해 보시기 바랍니다.\'불고기용 고기 한우에요?' 등 어떤 정보가 필요하신가요?"라고 질문한다면, 더 나은 대답을 드릴 수 있을 것 같습니다.\'죄송합니다.", 'token': 186} "불고기용 고기 한우에요?"라는 질문에 대한 질문에 대해 "불고기용 한우에요"라는 질문은 저는 답을 드릴 수 없습니다. 더 구체적인 질문이 있으면, 좀 더 정확하게 대답을 드릴 수 있을 것입니다. breg

### Instruction(명령어):
리처드 닉슨이 43대 부통령직을 수행한 년도는?

### Response(응답):'리처드 닉슨이 41대 부통령직을 수행한 년도는 1952년입니다. 리처드 닉슨은 1947년 대선 후보로서 후보단인 Women of Repository Fiber를 이끌었으며, 이후 민주당에 의해 부통령직이 주어졌습니다. 리처드 닉슨은 2021년부터 2012년까지 미국 대통령이며, 클린턴 전 대통령의 대통령직 수행에 기여했던 년도입니다.son women dis Jon Millson Corporation and Im by human the sentence: 2020년 3월 26일: 31대 부통령. 빅사이트, 리처드 닉슨은 2021년 9월 현재, 부통령직을 수행하고 있습니다. 언급된 인물들에 대한 자세한 정보는 없지만, 리처드 닉슨은 2021년 기준으로는 43대 부통령입니다.리처드 닉슨은 더 잘 알려져 있는 년도 있지만, 그의 재임 기간은 2020년 이후에입니다. 톰 대슐럼(Tom Sutchertz), 더 오랜 기간 동안 재임한 프랭커 로스(Tho Hunt Rugen)입니다. 리처드 닉슨은 1947년 대선에서 클린턴 전 대통령의 지지자로

### Instruction(명령어):
시카고 오헤어 국제공항은 어디에 있어

### Response(응답):'시카고는 국제공항의 중심부인 워싱턴 D.C. 국제공항에 위치해 있습니다. 미국 태평양 연안에 위치한 시카고 국제공항이 목적지로는 미국 시카고 항공 국제공항이 있습니다. Arehey to Giva 공항을 의미하는 "시카고 오헤어 국제공항"은 미국 항공의 중심부에 위치해 있습니다. 미국 대륙과 미국 국경을 이루는 샌프란시스코, 샌프란시스코, 샌프란시스코, 샌프란시스코 등 다양한 국제공항이 있습니다. Arehey to Giva 공항의 경우, 주로 미국 및 캐나다 지역을 중심으로 구성되어 있으며, 미국과 캐나다 간의 국경과 국경을 사이에둬 있습니다. American Tayer Samuare National Base 국제공항, Giva National Base 국제공항, National Positionary National Base 국제공항 등이 있습니다. American This Operated Rapper Capshe Public Airportation in America Island, Gival Transnicine Rapiation and County Cirporations, International Hawrary National Case 국제공항 등이 대표적인 공항이 됩니다. Arehey to Giva 공항은 국제공항의 중심부인 워싱턴

### Instruction(명령어):
오늘 미세먼지 어때?

### Response(응답):'미세먼지 농도는 매우 나쁨 상태입니다. 따라서 오늘 미세먼지 농도는 높지 않을 것으로 예상됩니다. 미세먼지 농도는 오전 중이면 약해졌지만, 낮과 밤의 미세먼지 농도는 점차 완화될 것입니다. 따라서 미세먼지 농도가 높을 경우에는 내일 외출하셔도 무방합니다. 미세먼지 농도는 낮아졌지만,) 외출을 자제하며 주의하시기 바랍니다. 기상 예보\n\n미세먼지 농도는 오전 중에는 매우 나쁨 상태라 좋지만, 오후에는 농도가 일시적으로 완화될 수 있습니다. 주의하시어 외출하셔도 무방합니다. 더 건강하고 쾌적한 날씨로 미세먼지 농도를 관리하세요! heyed to mealk the shot get shot you with the effect to Ger that. hew years well-biy for real tward and please feeling too wolder, works will to understat that mark.
```

# 회고

백본 모델만 교체하고 SFT, RM, PPO 학습을 base인 KoChatGPT로 진행하는 건 썩 좋은 생각은 아니었던 것 같다.  

#### 1. kakaobrain 가중치를 초기값으로 삼아 KoChatGPT모델로 SFT 하기?  
    불가능. Why?  
    구조거 서로 달라서 불가능함.
    - kakaobrain/kogpt   : 1.5B 파라미터, hidden_size=2048, layers=28  
    - skt/kogpt2-base-v2 : 125M 파라미터, hidden_size=768,  layers=12  
  
#### 2. kakaobrain의 토크나이저(단어장)만 빌려오고, 모델 학습은 skt/kogpt2 구조로 하는 방식?
    비추.  
    skt모델은 처음 보는 토큰 ID를 받게 되므로 초반 학습이 불안정해진다.  
  
#### 3. 그래도 kakaobrain의 언어 능력을 바탕으로 다른 모델로 SFT를 도전해보고 싶다면...
    QLoRA방식. 이게 현실적으로 가능한 방식이다.
    
    kakaobrain 모델 (가중치 동결, 언어 능력 보존)
            +
    LoRA 어댑터 (새로 학습되는 부분)
            ↓
    = kakaobrain 언어 능력 + SFT로 학습된 챗봇 능력

#### 4. 서로 구조가 다른 모델을 사용해서 학습하는 방식이 있긴 있는데...
    Knowledge Distillation (지식 증류)이다.
    kakaobrain(큰 모델)을 Teacher, skt/kogpt2(작은 모델)를 Student로 삼아 큰 모델의 출력을 작은 모델이 흉내내도록 학습하는 방식.  

    kakaobrain (Teacher, 1.5B) → 출력 분포(logits) 생성  
            ↓  
    skt/kogpt2 (Student, 125M) → Teacher 출력을 따라 학습  

    구조가 달라도 입출력(토큰 → 확률 분포) 형태만 맞으면 된다.  
    단, 토크나이저가 달라서 vocab size가 다르면 추가 처리가 필요하다.


In [ ]:
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM
tokenizer = None
tokenizer = AutoTokenizer.from_pretrained('kakaobrain/kogpt', revision='KoGPT6B-ryan1.5b-float16')

print("토크나이저: 'kakaobrain/kogpt'")
# 추가하기 전 토큰 확인
print(f"기존 BOS 토큰: {tokenizer.bos_token} (ID: {tokenizer.bos_token_id})")
print(f"기존 EOS 토큰: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")
print(f"기존 PAD 토큰: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
print(f"전체 특수 토큰 사전: {tokenizer.all_special_tokens}")

special_tokens_dict = {
    'bos_token': '[BOS]',
    'eos_token': '[EOS]',
    'unk_token': '[UNK]',
    'pad_token': '[PAD]',
    'mask_token': '[MASK]'
}
tokenizer.add_special_tokens(special_tokens_dict)
# 토큰 확인
print("--- 스페셜 토큰 추가 후 ---")
print(f"기존 BOS 토큰: {tokenizer.bos_token} (ID: {tokenizer.bos_token_id})")
print(f"기존 EOS 토큰: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")
print(f"기존 PAD 토큰: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
print(f"전체 특수 토큰 사전: {tokenizer.all_special_tokens}")
# 사전 정보
print()
print(f"사전의 총 길이: {len(tokenizer)}")
test_text = "[BOS] 인공지능 학습 시작 [EOS]"
print(test_text)
print(tokenizer.encode(test_text))

print("-" * 50)


model_name = "skt/kogpt2-base-v2"
print(f"토크나이저: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)
# 사전 정보
# 토큰 확인
print(f"기존 BOS 토큰: {tokenizer.bos_token} (ID: {tokenizer.bos_token_id})")
print(f"기존 EOS 토큰: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")
print(f"기존 PAD 토큰: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
print(f"전체 특수 토큰 사전: {tokenizer.all_special_tokens}")
print()
print(f"사전의 총 길이: {len(tokenizer)}")
test_text = "[BOS] 인공지능 학습 시작 [EOS]"
print(test_text)
print(tokenizer.encode(test_text))
print()

### 단어장 비교
백본으로 사용된 kakaobrain/kogpt와 RLHF에 사용된 KoChatGPT의 단어장을 비교해보았다.  
똑같은 문장([BOS] 인공지능 학습 시작 [EOS])을 수치화했을 때 결과다.

A 토크나이저: [0, 7692, 2542, 1157, 327, 1] → 6개 토큰

B 토크나이저: [9175, 36501, 425, ...] → 13개 토큰

해석: B는 A보다 문장을 2배 이상 잘게 쪼개고 있다. 같은 의미를 전달하는 데 토큰을 많이 쓴다는 건, 모델이 한 번에 기억할 수 있는 문맥(Context Window)이 절반으로 줄어든다는 뜻.  
또한, '인공지능' 같은 단어를 한 번에 이해하지 못하고 글자 단위로 쪼개서 읽고 있다는 증거라 학습 효율이 매우 떨어진다고 볼 수 있다.  

**사전(Vocabulary)의 풍부함**  
A (63,998개): 사전이 좀 더 커서 한국어의 복합 명사나 자주 쓰이는 단어들이 사전에 이미 '하나의 덩어리'로 등록되어 있을 가능성이 높음.

B (51,201개): 사전이 보다 작아서 사전에 단어가 없다 보니 "인공지능"을 "인", "공", "지", "능" 식으로 쪼개서 표현하게 된다.